# AOL_CV — LJK Scanner
Pipeline: Upload → Warp → Deteksi Nama/NIM/Tanggal/Jawaban → OCR → Scoring → EDA

In [ ]:
!pip install opencv-python-headless numpy matplotlib pillow easyocr scikit-learn scikit-image openpyxl -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from google.colab import files
import io, json, re
from IPython.display import display, HTML

## 1. Upload & Tampil Gambar LJK

In [ ]:
def show(img, title='', cmap=None, size=(10,8)):
    """Tampilkan satu gambar dengan judul."""
    plt.figure(figsize=size)
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_row(images, titles, size=(18,5)):
    """Tampilkan beberapa gambar dalam satu baris."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=size)
    if n == 1: axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def print_step(step, msg):
    """Print log step dengan format rapi."""
    icons = {'ok':'v','warn':'!','info':'i','err':'x','scan':'?','arrow':'->'}
    print(f"  [{step}] {msg}")

print('Upload foto LJK (JPG/PNG)...')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
img_bytes = uploaded[filename]

nparr  = np.frombuffer(img_bytes, np.uint8)
img_original = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

print(f'File: {filename}')
print(f'   Dimensi asli  : {img_original.shape[1]} x {img_original.shape[0]} px')
print(f'   Channel       : {img_original.shape[2]} (BGR)')

show(img_original, f'confirm input {filename}', size=(10,12))

## 2. Preprocessing & Deteksi Kontur Dokumen

In [ ]:
img = img_original.copy()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))

opened = cv2.morphologyEx(
    thresh,
    cv2.MORPH_OPEN,
    kernel,
    iterations=1
)

dilated = cv2.dilate(opened, kernel, iterations=2)

print_step('4', 'Morphology + Dilasi')

# =========================================
# FIND OUTER DOCUMENT
# =========================================

contours, _ = cv2.findContours(
    dilated,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

largest = None
max_area = 0

for cnt in contours:

    area = cv2.contourArea(cnt)

    if area > 5000:

        peri = cv2.arcLength(cnt, True)

        approx = cv2.approxPolyDP(
            cnt,
            0.02 * peri,
            True
        )

        if len(approx) == 4 and area > max_area:
            largest = approx
            max_area = area

print_step('5', 'Find Document Contour')

# =========================================
# DRAW CONTOUR
# =========================================

contour_img = img.copy()

if largest is not None:
    cv2.drawContours(
        contour_img,
        [largest],
        -1,
        (0,255,0),
        4
    )


show_row(
    [
        img,
        gray,
        thresh,
        contour_img
    ],
    [
        'Original',
        'Gray',
        'Threshold',
        'Contour'
    ],
    size=(25,5)
)

## 3. Deteksi 4 Kotak Sudut LJK

In [ ]:
print('='*55)
print('  DETEKSI 4 KOTAK SUDUT LJK')
print('='*55)

import numpy as np, cv2
import matplotlib.pyplot as plt

img_h, img_w = img.shape[:2]
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# -- Threshold gelap
_, dark = cv2.threshold(gray, 80, 255, cv2.THRESH_BINARY_INV)
k    = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, k, iterations=1)

# -- Cari semua kontur
contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

# -- Filter: solid + mendekati persegi
candidates = []
for c in contours:
    area    = cv2.contourArea(c)
    x,y,w,h = cv2.boundingRect(c)
    if w == 0 or h == 0: continue
    solidity = area / float(w * h)
    aspect   = w / float(h)
    min_a = img_w * img_h * 0.0003
    max_a = img_w * img_h * 0.04
    if min_a < area < max_a and 0.4 < aspect < 2.5 and solidity >= 0.65:
        candidates.append({
            'area': area, 'x':x,'y':y,'w':w,'h':h,
            'cx': x+w//2, 'cy': y+h//2
        })

# -- Dari semua kandidat, ambil 1 terdekat ke tiap sudut gambar
corners = {
    'TL': (0,        0       ),
    'TR': (img_w,    0       ),
    'BR': (img_w,    img_h   ),
    'BL': (0,        img_h   ),
}

selected = {}
for label, (tx, ty) in corners.items():
    if not candidates:
        break
    best = min(candidates,
               key=lambda a: (a['cx']-tx)**2 + (a['cy']-ty)**2)
    selected[label] = best

# -- Print hasil
print(f'\n  Total kandidat: {len(candidates)}')
print(f'\n  4 Kotak terpilih (terdekat ke tiap sudut):')
for label, a in selected.items():
    print(f'    {label}: cx={a["cx"]:4d}  cy={a["cy"]:4d}  '
          f'size={a["w"]}x{a["h"]}  area={a["area"]:.0f}')

# -- Visualisasi
COLORS = {'TL':(255,50,50),'TR':(50,50,255),
           'BR':(50,200,50),'BL':(255,165,0)}
vis = img.copy()

for label, a in selected.items():
    col = COLORS[label]
    cv2.rectangle(vis, (a['x'],a['y']),
                  (a['x']+a['w'], a['y']+a['h']), col, 3)
    cv2.circle(vis, (a['cx'],a['cy']), 10, col, -1)
    cv2.putText(vis, label,
                (a['cx']+12, a['cy']+8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, col, 2)

# garis penghubung
order = ['TL','TR','BR','BL']
for i in range(4):
    if order[i] in selected and order[(i+1)%4] in selected:
        p1 = (selected[order[i]]['cx'],       selected[order[i]]['cy'])
        p2 = (selected[order[(i+1)%4]]['cx'], selected[order[(i+1)%4]]['cy'])
        cv2.line(vis, p1, p2, (0,255,0), 2)

plt.figure(figsize=(7,10))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('4 Kotak Sudut LJK\nMerah=TL  Biru=TR  Hijau=BR  Orange=BL',
          fontweight='bold')
plt.axis('off'); plt.tight_layout(); plt.show()

## 4. Perspective Warp

In [ ]:
# -- Warp
if len(selected) == 4:
    pts = np.array([
        [selected['TL']['cx'], selected['TL']['cy']],
        [selected['TR']['cx'], selected['TR']['cy']],
        [selected['BR']['cx'], selected['BR']['cy']],
        [selected['BL']['cx'], selected['BL']['cy']],
    ], dtype='float32')

    s    = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    rect = np.zeros((4,2), dtype='float32')
    rect[0]=pts[np.argmin(s)];    rect[2]=pts[np.argmax(s)]
    rect[1]=pts[np.argmin(diff)]; rect[3]=pts[np.argmax(diff)]

    maxW = max(int(np.linalg.norm(rect[2]-rect[3])),
               int(np.linalg.norm(rect[1]-rect[0])))
    maxH = max(int(np.linalg.norm(rect[1]-rect[2])),
               int(np.linalg.norm(rect[0]-rect[3])))
    dst  = np.array([[0,0],[maxW-1,0],[maxW-1,maxH-1],[0,maxH-1]],
                    dtype='float32')

    M      = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(img, M, (maxW, maxH))
    warped = cv2.resize(warped, (1000, 1414))

    print(f'\n  Warp selesai -> 1000x1414 px')
    plt.figure(figsize=(8,11))
    plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    plt.title('Hasil Warp 1000x1414', fontweight='bold')
    plt.axis('off'); plt.tight_layout(); plt.show()

## 5. Milimeter Grid

In [ ]:
print('Membuat milimeter grid...')

import numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

img_h, img_w = warped.shape[:2]  # 1414 x 1000
rgb = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(14, 20))
ax.imshow(rgb, extent=[0, img_w, img_h, 0])

# -- Grid 10px - tipis abu
ax.set_xticks(range(0, img_w+1, 10), minor=True)
ax.set_yticks(range(0, img_h+1, 10), minor=True)
ax.grid(which='minor', color='#cccccc', linewidth=0.3, alpha=0.6)

# -- Grid 50px - sedang biru muda
ax.set_xticks(range(0, img_w+1, 50))
ax.set_yticks(range(0, img_h+1, 50))
ax.grid(which='major', color='#88aaff', linewidth=0.6, alpha=0.7)

# -- Grid 100px - tebal biru tua + label
for x in range(0, img_w+1, 100):
    ax.axvline(x=x, color='#2255cc', linewidth=1.0, alpha=0.9)
    ax.text(x+1, -10, str(x), fontsize=7, color='#2255cc',
            ha='left', va='bottom', fontweight='bold')

for y in range(0, img_h+1, 100):
    ax.axhline(y=y, color='#2255cc', linewidth=1.0, alpha=0.9)
    ax.text(-18, y, str(y), fontsize=7, color='#2255cc',
            ha='right', va='center', fontweight='bold')

ax.set_xlim(0, img_w)
ax.set_ylim(img_h, 0)
ax.tick_params(which='both', bottom=False, left=False,
               labelbottom=False, labelleft=False)
ax.set_title('MILIMETER GRID - abu=10px  biru muda=50px  biru tua=100px',
             fontsize=12, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('grid_view.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tersimpan: grid_view.png')
print('Download: files.download("grid_view.png")')

## 6. Deteksi Nama

In [ ]:
print('='*55)
print('  DETEKSI NAMA - RELATIVE DARKNESS PER KOLOM')
print('='*55)

import numpy as np, cv2
import matplotlib.pyplot as plt

# ================================================
#  ROI NAMA
# ================================================
X1, Y1, X2, Y2 = 40, 270, 520, 850
ALPHABET = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
NUM_COLS = 20

roi      = warped[Y1:Y2, X1:X2]
roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
roi_h, roi_w = roi_gray.shape

# -- CLAHE: normalisasi lighting lokal
# Ini kunci utama - tiap area dinormalisasi sendiri
clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
roi_eq   = clahe.apply(roi_gray)

# Invert: bubble terisi = pixel gelap -> setelah invert jadi terang
roi_inv  = cv2.bitwise_not(roi_eq)

# -- Auto-detect bubble area (skip header)
blur     = cv2.GaussianBlur(roi_gray, (3,3), 0)
_, bw    = cv2.threshold(blur, 0, 255,
                         cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
row_proj = np.sum(bw, axis=1) / 255

bubble_start = next((i for i,v in enumerate(row_proj)
                     if v > roi_w * 0.05), 0)
bubble_end   = next((i for i in range(roi_h-1,0,-1)
                     if row_proj[i] > roi_w * 0.05), roi_h)

bubble_h = bubble_end - bubble_start
row_h    = bubble_h / 26
col_w    = roi_w / NUM_COLS

print(f'  Bubble area : y={bubble_start}->{bubble_end}  '
      f'cell {col_w:.1f}x{row_h:.1f}px')

# -- Scan: per kolom, hitung mean brightness tiap baris
# Pakai roi_inv -> bubble terisi = nilai TINGGI
nama_result = []
density_map = np.zeros((26, NUM_COLS), dtype=float)
raw_scores  = np.zeros((26, NUM_COLS), dtype=float)

for col in range(NUM_COLS):
    x0 = int(col     * col_w) + 2   # +2 margin kiri
    x1 = int((col+1) * col_w) - 2   # -2 margin kanan
    scores = []

    for row in range(26):
        y0   = bubble_start + int(row     * row_h) + 2
        y1   = bubble_start + int((row+1) * row_h) - 2
        cell = roi_inv[y0:y1, x0:x1]
        if cell.size == 0:
            scores.append(0.0); continue
        # Mean brightness di area tengah cell (ignore tepi)
        ch, cw = cell.shape
        cy0,cy1 = ch//4, 3*ch//4
        cx0,cx1 = cw//4, 3*cw//4
        center  = cell[cy0:cy1, cx0:cx1]
        s = float(np.mean(center)) if center.size > 0 else 0.0
        scores.append(s)
        raw_scores[row, col] = s

    # Normalisasi per kolom: z-score
    arr  = np.array(scores)
    mean = arr.mean()
    std  = arr.std() + 1e-6
    z    = (arr - mean) / std

    density_map[:, col] = z

    # Pilih baris dengan z-score tertinggi
    best_row   = int(np.argmax(z))
    best_z     = z[best_row]
    second_z   = sorted(z)[-2]

    # Dianggap terisi jika:
    # 1. z-score terbaik > 1.5 (jauh di atas rata-rata kolom)
    # 2. Selisih best vs second cukup besar (tidak ambiguous)
    if best_z > 1.2 and (best_z - second_z) > 0.5:
        nama_result.append(ALPHABET[best_row])
    else:
        nama_result.append('_')

nama_str = ''.join(nama_result).replace('_',' ').strip()

print(f'\n  Per kolom : {"" .join(nama_result)}')
print(f'\n  NAMA : {nama_str}')

# -- Visualisasi
fig, axes = plt.subplots(1, 3, figsize=(22, 9))

# 1. ROI asli + highlight
vis = roi.copy()
for col in range(NUM_COLS):
    x0 = int(col * col_w)
    x1 = int((col+1) * col_w)
    cv2.line(vis,(x0,bubble_start),(x0,bubble_end),(255,140,0),1)
    char = nama_result[col]
    if char != '_':
        row = ALPHABET.index(char)
        ry0 = bubble_start + int(row * row_h)
        ry1 = bubble_start + int((row+1) * row_h)
        cv2.rectangle(vis,(x0+1,ry0),(x1-1,ry1),(0,255,0),2)
        cv2.putText(vis, char,(x0+2,ry0+int(row_h)-2),
                    cv2.FONT_HERSHEY_SIMPLEX,0.35,(0,220,0),1)

axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Hasil Deteksi\n-> {nama_str}',
                  fontweight='bold', fontsize=11)
axes[0].axis('off')

# 2. ROI setelah CLAHE
axes[1].imshow(roi_eq, cmap='gray')
axes[1].set_title('Setelah CLAHE\n(lighting diratakan)',
                  fontweight='bold', fontsize=11)
axes[1].axis('off')

# 3. Heatmap z-score
im = axes[2].imshow(density_map, aspect='auto', cmap='RdYlGn',
                    interpolation='nearest', vmin=-3, vmax=3)
axes[2].set_yticks(range(26))
axes[2].set_yticklabels(ALPHABET, fontsize=8)
axes[2].set_xticks(range(NUM_COLS))
axes[2].set_xticklabels(
    [f'{i+1}\n{nama_result[i]}' for i in range(NUM_COLS)],
    fontsize=8
)
axes[2].set_title('Z-score per Cell\n(hijau=paling gelap di kolom)',
                  fontweight='bold', fontsize=11)
plt.colorbar(im, ax=axes[2])

plt.suptitle(f'NAMA: {nama_str}',
             fontsize=15, fontweight='bold', color='darkgreen', y=1.01)
plt.tight_layout()
plt.show()

## 7. Full Detection: NIM + Tanggal + Jenis Ujian + Jawaban

In [ ]:
print('='*55)
print('  FULL DETECTION: NIM + TANGGAL + JENIS UJIAN + JAWABAN')
print('='*55)

import numpy as np, cv2, json
import matplotlib.pyplot as plt

ALPHABET = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
CHOICES  = ['A','B','C','D','E']
JENIS_UJIAN_LABELS = [
    'UJIAN TENGAH SEMESTER',
    'UJIAN AKHIR SEMESTER',
    'KUIS / LATIHAN SOAL',
    'LAINNYA',
]

# ================================================
#  ROI KONFIGURASI
# ================================================
ROI_NIM        = (550, 270, 790, 500)
ROI_TANGGAL    = (820, 270, 970, 500)
ROI_JENIS_UJIAN= (550, 750, 595, 860)   # hanya bubble, potong teks

ROI_JAWABAN = [
    ('1-10',    70,  930, 190, 1150,  1),
    ('11-20',   70, 1160, 190, 1390, 11),
    ('21-30',  250,  930, 380, 1150, 21),
    ('31-40',  259, 1160, 380, 1390, 31),
    ('41-50',  450,  930, 580, 1150, 41),
    ('51-60',  450, 1160, 580, 1390, 51),
    ('61-70',  650,  930, 780, 1150, 61),
    ('71-80',  650, 1160, 780, 1390, 71),
    ('81-90',  830,  930, 950, 1150, 81),
    ('91-100', 830, 1160, 950, 1390, 91),
]

# ================================================
#  HELPER
# ================================================
def apply_clahe(gray):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(gray)

def get_bubble_range(gray):
    """Auto-detect baris pertama & terakhir bubble."""
    blur = cv2.GaussianBlur(gray,(3,3),0)
    _,bw = cv2.threshold(blur,0,255,
                         cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    rp   = np.sum(bw,axis=1)/255
    w    = gray.shape[1]
    b0   = next((i for i,v in enumerate(rp) if v>w*0.03), 0)
    b1   = next((i for i in range(len(rp)-1,0,-1)
                 if rp[i]>w*0.03), gray.shape[0])
    return b0, b1

def scan_grid(gray, num_cols, num_rows, labels,
              z_thresh=1.2, z_gap=0.5, per_row=False):
    """
    Scan grid num_cols x num_rows.
    per_row=False -> tiap kolom = 1 item (untuk NIM/nama)
    per_row=True  -> tiap baris = 1 item, kolom = pilihan (untuk jawaban)
    Return: results list, density_map, b0, b1, row_h, col_w
    """
    eq  = apply_clahe(gray)
    inv = cv2.bitwise_not(eq)
    h,w = gray.shape
    b0,b1 = get_bubble_range(gray)
    bh    = b1 - b0
    row_h = bh / num_rows
    col_w = w  / num_cols

    density_map = np.zeros((num_rows, num_cols), dtype=float)
    raw         = np.zeros((num_rows, num_cols), dtype=float)

    for r in range(num_rows):
        for c in range(num_cols):
            y0 = b0 + int(r*row_h)+2;  y1b = b0+int((r+1)*row_h)-2
            x0 = int(c*col_w)+2;       x1b = int((c+1)*col_w)-2
            cell = inv[y0:y1b, x0:x1b]
            if cell.size==0: continue
            ch,cw = cell.shape
            cx = cell[ch//4:3*ch//4, cw//4:3*cw//4]
            raw[r,c] = float(np.mean(cx)) if cx.size>0 else 0.0

    results = []
    if not per_row:
        # per kolom -> tiap kolom = 1 karakter
        for c in range(num_cols):
            arr  = raw[:,c]
            mean,std = arr.mean(), arr.std()+1e-6
            z    = (arr-mean)/std
            density_map[:,c] = z
            best = int(np.argmax(z))
            bz   = z[best]; sz = sorted(z)[-2]
            if bz>z_thresh and (bz-sz)>z_gap and best<len(labels):
                results.append(labels[best])
            else:
                results.append(None)
    else:
        # per baris -> tiap baris = 1 soal
        for r in range(num_rows):
            arr  = raw[r,:]
            mean,std = arr.mean(), arr.std()+1e-6
            z    = (arr-mean)/std
            density_map[r,:] = z
            best = int(np.argmax(z))
            bz   = z[best]; sz = sorted(z)[-2]
            if bz>z_thresh and (bz-sz)>z_gap and best<len(labels):
                results.append(labels[best])
            else:
                results.append(None)

    return results, density_map, b0, b1, row_h, col_w

def show_detection(roi_bgr, results, density_map,
                   b0, b1, row_h, col_w,
                   row_labels, col_labels,
                   title, per_row=False):
    """Tampilkan 3-panel: ROI highlight + CLAHE + heatmap z-score."""
    gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    eq   = apply_clahe(gray)
    h,w  = gray.shape
    num_rows = len(row_labels)
    num_cols = len(col_labels)

    # Panel 1: highlight bubble terpilih
    vis = roi_bgr.copy()
    # garis grid
    for c in range(num_cols):
        cv2.line(vis,(int(c*col_w),b0),(int(c*col_w),b1),(255,140,0),1)
    for r in range(num_rows):
        cv2.line(vis,(0,int(b0+r*row_h)),(w,int(b0+r*row_h)),(200,200,200),1)

    if not per_row:
        for c,res in enumerate(results):
            if res is None: continue
            r   = row_labels.index(res) if res in row_labels else -1
            if r<0: continue
            x0,x1b = int(c*col_w), int((c+1)*col_w)
            y0,y1b = int(b0+r*row_h), int(b0+(r+1)*row_h)
            cv2.rectangle(vis,(x0,y0),(x1b,y1b),(0,255,0),2)
            cv2.putText(vis,res,(x0+2,y1b-2),
                        cv2.FONT_HERSHEY_SIMPLEX,0.3,(0,220,0),1)
        result_str = ''.join(r or '_' for r in results)
    else:
        for r,res in enumerate(results):
            if res is None: continue
            c   = col_labels.index(res) if res in col_labels else -1
            if c<0: continue
            x0,x1b = int(c*col_w), int((c+1)*col_w)
            y0,y1b = int(b0+r*row_h), int(b0+(r+1)*row_h)
            cv2.rectangle(vis,(x0,y0),(x1b,y1b),(0,255,0),2)
            cv2.putText(vis,res,(x0+2,y1b-2),
                        cv2.FONT_HERSHEY_SIMPLEX,0.3,(0,220,0),1)
        result_str = ' '.join(r or '-' for r in results)

    fig, axes = plt.subplots(1,3, figsize=(18,6))

    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Hasil Deteksi\n-> {result_str}',
                      fontweight='bold', fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(eq, cmap='gray')
    axes[1].set_title('Setelah CLAHE\n(lighting diratakan)',
                      fontweight='bold', fontsize=10)
    axes[1].axis('off')

    im = axes[2].imshow(density_map, aspect='auto', cmap='RdYlGn',
                        interpolation='nearest', vmin=-3, vmax=3)
    axes[2].set_yticks(range(num_rows))
    axes[2].set_yticklabels(row_labels, fontsize=7)
    axes[2].set_xticks(range(num_cols))
    axes[2].set_xticklabels(col_labels, fontsize=7)
    axes[2].set_title('Z-score per Cell\n(hijau=paling gelap)',
                      fontweight='bold', fontsize=10)
    plt.colorbar(im, ax=axes[2])

    plt.suptitle(title, fontsize=13, fontweight='bold',
                 color='darkgreen', y=1.01)
    plt.tight_layout()
    plt.show()

    return result_str

# ================================================
#  DETEKSI NIM
# ================================================
print('\n-- NIM')
x1,y1,x2,y2 = ROI_NIM
roi_nim  = cv2.cvtColor(warped[y1:y2,x1:x2], cv2.COLOR_BGR2GRAY)
r_nim, dm_nim, b0,b1,rh,cw = scan_grid(
    roi_nim, num_cols=10, num_rows=10,
    labels=[str(i) for i in range(10)], per_row=False)
nim_str = show_detection(
    warped[y1:y2,x1:x2], r_nim, dm_nim, b0,b1,rh,cw,
    row_labels=[str(i) for i in range(10)],
    col_labels=[f'D{i+1}' for i in range(10)],
    title=f'NIM: {"" .join(r or "_" for r in r_nim)}',
    per_row=False)
nim_final = ''.join(r or '_' for r in r_nim)
print(f'  NIM : {nim_final}')

# ================================================
#  DETEKSI TANGGAL
# ================================================
print('\n-- TANGGAL')
x1,y1,x2,y2 = ROI_TANGGAL
roi_tgl  = cv2.cvtColor(warped[y1:y2,x1:x2], cv2.COLOR_BGR2GRAY)
# Hitung jumlah kolom dari lebar ROI
tgl_cols = 6
r_tgl, dm_tgl, b0,b1,rh,cw = scan_grid(
    roi_tgl, num_cols=tgl_cols, num_rows=10,
    labels=[str(i) for i in range(10)], per_row=False)
tgl_final = ''.join(r or '_' for r in r_tgl)
show_detection(
    warped[y1:y2,x1:x2], r_tgl, dm_tgl, b0,b1,rh,cw,
    row_labels=[str(i) for i in range(10)],
    col_labels=[f'D{i+1}' for i in range(tgl_cols)],
    title=f'TANGGAL: {tgl_final}',
    per_row=False)
print(f'  TANGGAL : {tgl_final}')

# ================================================
#  DETEKSI JENIS UJIAN
# ================================================
print('\n-- JENIS UJIAN')
x1,y1,x2,y2 = ROI_JENIS_UJIAN
roi_ju_bgr = warped[y1:y2,x1:x2]
roi_ju     = cv2.cvtColor(roi_ju_bgr, cv2.COLOR_BGR2GRAY)

# Jenis ujian: 4 bubble tersusun VERTIKAL (1 kolom, 4 baris)
# scan_grid per_row=False, num_cols=1, num_rows=4
r_ju, dm_ju, b0,b1,rh,cw = scan_grid(
    roi_ju, num_cols=1, num_rows=4,
    labels=JENIS_UJIAN_LABELS, per_row=False)
jenis_final = r_ju[0] if r_ju else None

show_detection(
    roi_ju_bgr, r_ju, dm_ju, b0,b1,rh,cw,
    row_labels=['UTS','UAS','KUIS','LAIN'],
    col_labels=['Pilihan'],
    title=f'JENIS UJIAN: {jenis_final}',
    per_row=False)
print(f'  JENIS UJIAN : {jenis_final}')

# ================================================
#  INPUT: BERAPA SOAL YANG DICEK?
# ================================================
print('\n-- KONFIGURASI JAWABAN')
TOTAL_SOAL = int(input('  Masukkan jumlah soal (1-100): '))
TOTAL_SOAL = max(1, min(100, TOTAL_SOAL))
print(f'  -> Akan mendeteksi {TOTAL_SOAL} soal')

# ================================================
#  DETEKSI JAWABAN - hanya blok yang diperlukan
# ================================================
print('\n-- JAWABAN')
all_answers = {}
soal_done   = 0

for label, x1,y1,x2,y2, q_start in ROI_JAWABAN:
    if soal_done >= TOTAL_SOAL:
        break

    # Berapa soal di blok ini yang perlu diproses
    soal_di_blok = min(10, TOTAL_SOAL - soal_done)

    roi_bgr  = warped[y1:y2, x1:x2]
    roi_gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)

    r_blok, dm_blok, b0,b1,rh,cw = scan_grid(
        roi_gray, num_cols=5, num_rows=10,
        labels=CHOICES, per_row=True)

    # Potong hanya soal yang aktif
    r_aktif = r_blok[:soal_di_blok]

    show_detection(
        roi_bgr, r_aktif + [None]*(10-soal_di_blok),
        dm_blok, b0,b1,rh,cw,
        row_labels=[str(q_start+i) for i in range(10)],
        col_labels=CHOICES,
        title=f'JAWABAN Soal {q_start}-{q_start+soal_di_blok-1}  ->  '
              + ' '.join(f'{q_start+i}:{r_aktif[i] or "-"}'
                         for i in range(soal_di_blok)),
        per_row=True)

    for i, ans in enumerate(r_aktif):
        q = q_start + i
        all_answers[q] = ans
        print(f'  Soal {q:3d}: {ans or "-"}', end='  ')
    print()

    soal_done += soal_di_blok

# ================================================
#  RINGKASAN
# ================================================
print('\n' + '='*55)
print('  HASIL AKHIR')
print('='*55)
print(f'  NIM         : {nim_final}')
print(f'  TANGGAL     : {tgl_final}')
print(f'  JENIS UJIAN : {jenis_final}')
print()

result_json = {
    'nim'        : nim_final,
    'tanggal'    : tgl_final,
    'jenis_ujian': jenis_final,
    'jawaban'    : {str(k): v for k,v in sorted(all_answers.items())}
}
print(json.dumps(result_json, indent=2, ensure_ascii=False))

## 8. Master ROI & OCR Setup

In [ ]:
"""### Kaggle (SVM + HOG)"""

# MASTER ROI
ALL_ROIS = {
    'NAMA_MATA_KULIAH': (535, 550, 965, 620),
    'KODE_KELAS':       (535, 658, 780, 710),
    'RUANGAN':          (820, 658, 965, 710),
    'NO_MEJA':          (820, 770, 965, 865),
}

HANDWRITING_ROIS = [
    (k, *ALL_ROIS[k])
    for k in ('NAMA_MATA_KULIAH', 'KODE_KELAS', 'RUANGAN', 'NO_MEJA')
]

print('Master ROI loaded.')
print(f'  OCR fields : {[r[0] for r in HANDWRITING_ROIS]}')

## 9. SVM + HOG — Collect Dataset & Train

In [ ]:
import cv2, numpy as np, os, pickle
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog

SAVE_DIR   = 'dataset/chars'
MODEL_PATH = 'svm_emnist.pkl'
CHAR_H, CHAR_W = 32, 32
os.makedirs(SAVE_DIR, exist_ok=True)


CHAR_ROIS = {
    'A': (825, 668, 852, 710),
    '1': (853, 668, 868, 710),
    '7': (868, 668, 893, 710),
    '0': (893, 668, 923, 710),
    '9': (923, 668, 955, 710),
    'L': (570, 658, 607, 710),
    'K': (607, 658, 648, 710),
    'O': (648, 658, 695, 710),
    '2': (695, 658, 740, 710),
    '1': (855, 775, 878, 855),
    '7': (878, 775, 920, 855),
}


def augment_char(img, n=30):
    h, w    = img.shape
    results = [img]
    for _ in range(n):
        aug = img.copy()

        #Rotation +-15 degree
        angle = np.random.uniform(-15, 15)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h), borderValue=255)

        #Scale 85-115%
        sc    = np.random.uniform(0.85, 1.15)
        aug   = cv2.resize(aug, (max(1,int(w*sc)), max(1,int(h*sc))))
        aug   = cv2.resize(aug, (w, h))

        # Noise
        noise = np.random.normal(0, 8, aug.shape).astype(np.int16)
        aug   = np.clip(aug.astype(np.int16)+noise, 0, 255).astype(np.uint8)

        #Blur
        if np.random.rand() > 0.5:
            aug = cv2.GaussianBlur(aug, (3,3), 0)

        #Shear horizontal
        dx   = np.random.uniform(-4, 4)
        pts1 = np.float32([[0,0],[w,0],[0,h]])
        pts2 = np.float32([[dx,0],[w+dx,0],[0,h]])
        M    = cv2.getAffineTransform(pts1, pts2)
        aug  = cv2.warpAffine(aug, M, (w,h), borderValue=255)

        results.append(aug)
    return results

def extract_hog(img):
    return hog(img, orientations=9,
               pixels_per_cell=(8,8),
               cells_per_block=(2,2),
               block_norm='L2-Hys')

def preprocess_char(crop_bgr):
    b, g, r = cv2.split(crop_bgr.astype(np.float32))
    gray    = np.clip(r*0.8+g*0.15+b*0.05, 0, 255).astype(np.uint8)
    p_lo, p_hi = np.percentile(gray, 5), np.percentile(gray, 95)
    if p_hi > p_lo:
        gray = np.clip((gray.astype(np.float32)-p_lo)
                       /(p_hi-p_lo)*255, 0, 255).astype(np.uint8)
    _, bw = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    return cv2.resize(bw, (CHAR_W, CHAR_H), interpolation=cv2.INTER_AREA)


def collect_from_uploads():
    print('Upload semua foto LJK (bisa sekaligus)...')
    uploaded = files.upload()
    total    = 0

    for fname, data in uploaded.items():
        nparr   = np.frombuffer(data, np.uint8)
        raw_img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if raw_img is None:
            print(f'  Gagal baca: {fname}'); continue

        try:
            img_h, img_w = raw_img.shape[:2]
            gray = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
            _, dark = cv2.threshold(gray, 80, 255, cv2.THRESH_BINARY_INV)
            k    = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
            dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, k)
            contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
            candidates = []
            for c in contours:
                area = cv2.contourArea(c)
                x,y,w,h = cv2.boundingRect(c)
                if w==0 or h==0: continue
                sol = area/float(w*h)
                asp = w/float(h)
                mn  = img_w*img_h*0.0003
                mx  = img_w*img_h*0.04
                if mn<area<mx and 0.4<asp<2.5 and sol>=0.65:
                    candidates.append({'cx':x+w//2,'cy':y+h//2,
                                       'x':x,'y':y,'w':w,'h':h})
            corners = {'TL':(0,0),'TR':(img_w,0),
                       'BR':(img_w,img_h),'BL':(0,img_h)}
            sel = {}
            for lbl,(tx,ty) in corners.items():
                if candidates:
                    sel[lbl] = min(candidates,
                        key=lambda a:(a['cx']-tx)**2+(a['cy']-ty)**2)
            if len(sel)==4:
                pts = np.float32([
                    [sel['TL']['cx'],sel['TL']['cy']],
                    [sel['TR']['cx'],sel['TR']['cy']],
                    [sel['BR']['cx'],sel['BR']['cy']],
                    [sel['BL']['cx'],sel['BL']['cy']]])
                s=pts.sum(1); d=np.diff(pts,axis=1)
                rect=np.zeros((4,2),dtype='float32')
                rect[0]=pts[np.argmin(s)]; rect[2]=pts[np.argmax(s)]
                rect[1]=pts[np.argmin(d)]; rect[3]=pts[np.argmax(d)]
                mW=max(int(np.linalg.norm(rect[2]-rect[3])),
                       int(np.linalg.norm(rect[1]-rect[0])))
                mH=max(int(np.linalg.norm(rect[1]-rect[2])),
                       int(np.linalg.norm(rect[0]-rect[3])))
                dst=np.float32([[0,0],[mW-1,0],[mW-1,mH-1],[0,mH-1]])
                M  =cv2.getPerspectiveTransform(rect,dst)
                page=cv2.warpPerspective(raw_img,M,(mW,mH))
                page=cv2.resize(page,(1000,1414))
            else:
                page = raw_img
        except Exception:
            page = raw_img

        #Crop tiap karakter
        saved_this = 0
        for char, (x1,y1,x2,y2) in CHAR_ROIS.items():
            crop = page[y1:y2, x1:x2]
            if crop.size == 0: continue
            bw   = preprocess_char(crop)
            idx  = len([f for f in os.listdir(SAVE_DIR)
                        if f.startswith(f'{char}__')])
            cv2.imwrite(f'{SAVE_DIR}/{char}__{idx:04d}.png', bw)
            saved_this += 1

        print(f'  {fname}: {saved_this} karakter disimpan')
        total += saved_this

    print(f'\n  Total tersimpan: {total} karakter')
    print(f'  Distribusi:')
    for char in sorted(CHAR_ROIS.keys()):
        n = len([f for f in os.listdir(SAVE_DIR)
                 if f.startswith(f'{char}__')])
        print(f'    {char}: {n} sampel')

def retrain_svm():
    X, y = [], []
    chars_found = []

    for fname in os.listdir(SAVE_DIR):
        if not fname.endswith('.png'): continue
        char = fname.split('__')[0]
        img  = cv2.imread(os.path.join(SAVE_DIR, fname),
                          cv2.IMREAD_GRAYSCALE)
        if img is None: continue

        augmented = augment_char(img, n=30)
        for aug in augmented:
            feat = extract_hog(aug)
            X.append(feat)
            y.append(char)
        chars_found.append(char)

    if not X:
        print('  Tidak ada data!'); return None

    X = np.array(X)
    y = np.array(y)
    print(f'  Training SVM: {len(X)} sampel, '
          f'{len(set(y))} kelas: {sorted(set(y))}')

    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('svm',    SVC(kernel='rbf', C=10, gamma='scale',
                       random_state=42))
    ])
    clf.fit(X, y)

    bundle = {'clf': clf, 'chars': sorted(set(y))}
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(bundle, f)
    print(f'  Model tersimpan -> {MODEL_PATH}')
    print(f'  Karakter: {sorted(set(y))}')
    return bundle

def preview_dataset():
    chars = sorted(set(f.split('__')[0]
                       for f in os.listdir(SAVE_DIR)
                       if f.endswith('.png')))
    if not chars:
        print('Dataset kosong.'); return

    fig, axes = plt.subplots(len(chars), 5,
                             figsize=(12, 2*len(chars)))
    if len(chars) == 1: axes = [axes]

    for i, char in enumerate(chars):
        files_c = sorted([f for f in os.listdir(SAVE_DIR)
                          if f.startswith(f'{char}__')])[:5]
        for j in range(5):
            axes[i][j].axis('off')
            if j < len(files_c):
                img = cv2.imread(
                    os.path.join(SAVE_DIR, files_c[j]),
                    cv2.IMREAD_GRAYSCALE)
                axes[i][j].imshow(img, cmap='gray')
                if j == 0:
                    axes[i][j].set_title(f'"{char}"',
                                         fontsize=11,
                                         fontweight='bold')

    plt.suptitle('Dataset - 5 sampel pertama per karakter',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()

collect_from_uploads()

preview_dataset()

print('\nRetrain SVM...')
bundle = retrain_svm()
print('\nSelesai')

## 10. Reset & Train dengan EMNIST

In [ ]:
import os

MODEL_PATH = 'svm_emnist.pkl'
if os.path.exists(MODEL_PATH):
    os.remove(MODEL_PATH)
    print(f'Deleted existing model file: {MODEL_PATH}')
else:
    print(f'Model file not found: {MODEL_PATH}')

import warnings
warnings.filterwarnings('ignore')

import tensorflow_datasets as tfds
import numpy as np, cv2, pickle, os, re
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog

CHAR_H, CHAR_W = 32, 32
MODEL_PATH     = 'svm_emnist.pkl'
SAVE_DIR       = 'dataset/chars'

HANDWRITING_ROIS = [
    (k, *ALL_ROIS[k])
    for k in ('NAMA_MATA_KULIAH', 'KODE_KELAS', 'RUANGAN', 'NO_MEJA')
]

N_CHARS = {
    'RUANGAN'         : 5,
    'KODE_KELAS'      : 4,
    'NO_MEJA'         : 2,
    'NAMA_MATA_KULIAH': 13,
}

ALL_CHARS = sorted(
    list('0123456789') +
    list('ABCDEFGHIJKLMNOPQRSTUVWXYZ') +
    list('abcdefghijklmnopqrstuvwxyz')
)

def extract_hog(img):
    return hog(img,
               orientations=9,
               pixels_per_cell=(8, 8),
               cells_per_block=(2, 2),
               block_norm='L2-Hys')

def augment_char(img, n=30):
    h, w    = img.shape
    results = [img]
    for _ in range(n):
        aug = img.copy()

        # Rotation +-15
        angle = np.random.uniform(-15, 15)
        M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        aug   = cv2.warpAffine(aug, M, (w, h), borderValue=255)

        # Scale 85-115%
        sc    = np.random.uniform(0.85, 1.15)
        aug   = cv2.resize(aug, (max(1,int(w*sc)), max(1,int(h*sc))))
        aug   = cv2.resize(aug, (w, h))

        # Noise
        noise = np.random.normal(0, 8, aug.shape).astype(np.int16)
        aug   = np.clip(aug.astype(np.int16)+noise, 0, 255).astype(np.uint8)

        # Slight blur
        if np.random.rand() > 0.5:
            aug = cv2.GaussianBlur(aug, (3,3), 0)

        # Shear horizontal
        dx   = np.random.uniform(-4, 4)
        pts1 = np.float32([[0,0],[w,0],[0,h]])
        pts2 = np.float32([[dx,0],[w+dx,0],[0,h]])
        M    = cv2.getAffineTransform(pts1, pts2)
        aug  = cv2.warpAffine(aug, M, (w,h), borderValue=255)

        results.append(aug)
    return results

def label_to_char(label):
    label = int(label)
    if label < 10:
        return str(label)
    elif label < 36:
        return chr(ord('A') + label - 10)
    else:
        return chr(ord('a') + label - 36)

def load_or_train():
    if os.path.exists(MODEL_PATH):
        print('Model ditemukan, loading...')
        with open(MODEL_PATH, 'rb') as f:
            bundle = pickle.load(f)
        if 'chars' not in bundle:
            bundle['chars'] = ALL_CHARS
        print(f'  Karakter: {len(bundle["chars"])} kelas: {sorted(bundle["chars"])}'
)
        return bundle

    print('Training model with EMNIST ByClass AND custom uploaded characters (if any).')
    print('  This will take a while (~10-15 minutes or more depending on sample size and resources).')

    X_all, y_all = [], []
    current_chars = set()

    if os.path.exists(SAVE_DIR) and os.listdir(SAVE_DIR):
        print(f'Collecting custom characters from {SAVE_DIR}...')
        for fname in os.listdir(SAVE_DIR):
            if not fname.endswith('.png'): continue
            char = fname.split('__')[0]
            img  = cv2.imread(os.path.join(SAVE_DIR, fname), cv2.IMREAD_GRAYSCALE)
            if img is None: continue


            augmented_imgs = augment_char(img, n=10)

            for aug_img in augmented_imgs:
                feat = extract_hog(aug_img)
                X_all.append(feat)
                y_all.append(char)
            current_chars.add(char)
        print(f'  Collected {len(X_all)} custom samples for {len(current_chars)} characters: {sorted(list(current_chars))}')
    else:
        print(f'  No custom characters found in {SAVE_DIR}. Skipping custom sample collection.')


    print('Downloading + loading EMNIST ByClass (~624 MB) and combining with collected samples...')
    ds = tfds.load('emnist/byclass', split='train', as_supervised=True)

    MAX_PER_CLS_EMNIST = 100
    emnist_counts = {c: 0 for c in ALL_CHARS}

    for img, label in tfds.as_numpy(ds):
        char = label_to_char(label)
        if char not in ALL_CHARS: continue
        if emnist_counts[char] >= MAX_PER_CLS_EMNIST:
            continue

        emnist_img = np.rot90(img.squeeze(), k=3)
        emnist_img = np.fliplr(emnist_img)
        emnist_img = cv2.resize(emnist_img, (CHAR_W, CHAR_H))
        _, emnist_bw = cv2.threshold(emnist_img, 50, 255, cv2.THRESH_BINARY)

        feat = extract_hog(emnist_bw)
        X_all.append(feat)
        y_all.append(char)
        current_chars.add(char)
        emnist_counts[char] += 1

    if not X_all:
        print('No data to train with after collecting custom and EMNIST samples!'); return None

    X_all = np.array(X_all)
    y_all = np.array(y_all)
    print(f'\n  Total samples for training : {len(X_all)}')
    print(f'  Total unique characters    : {len(current_chars)}: {sorted(list(current_chars))}')


    print('Training SVM classifier...')
    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('svm',    SVC(kernel='rbf', C=10, gamma='scale',
                       random_state=42, verbose=False))
    ])
    clf.fit(X_all, y_all)

    bundle = {'clf': clf, 'chars': sorted(list(current_chars))}
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(bundle, f)
    print(f'\nModel saved -> {MODEL_PATH}')
    print(f'Characters in model ({len(bundle["chars"])})\n    {bundle["chars"]}')
    return bundle

def preprocess(roi_bgr):
    b, g, r = cv2.split(roi_bgr.astype(np.float32))
    gray = np.clip(r*0.8 + g*0.15 + b*0.05, 0, 255).astype(np.uint8)

    h, w = gray.shape
    gray = gray[int(h*0.08):h-int(h*0.08),
                int(w*0.08):w-int(w*0.08)]
    if gray.size == 0: return None

    p_lo, p_hi = np.percentile(gray, 10), np.percentile(gray, 90)
    if p_hi > p_lo:
        gray = np.clip((gray.astype(np.float32)-p_lo)
                       /(p_hi-p_lo)*255, 0, 255).astype(np.uint8)

    gray = cv2.filter2D(gray, -1,
                        np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))

    h, w  = gray.shape
    scale = max(1.0, 64/h)
    return cv2.resize(gray, (int(w*scale), int(h*scale)),
                      interpolation=cv2.INTER_CUBIC)

def binarize(gray):
    _, bw = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return bw

def segment_fixed(binary, n_chars):
    h, w   = binary.shape
    char_w = w // n_chars
    chars  = []

    for i in range(n_chars):
        x0   = i * char_w
        x1   = (i+1) * char_w if i < n_chars-1 else w
        crop = binary[:, x0:x1]

        row_proj = np.sum(crop, axis=1)
        rows     = np.where(row_proj > 0)[0]
        if len(rows) > 0:
            y0, y1 = rows[0], rows[-1]+1
            crop   = crop[y0:y1, :]

        if crop.size == 0:
            crop = np.zeros((CHAR_H, CHAR_W), dtype=np.uint8)

        crop = cv2.resize(crop, (CHAR_W, CHAR_H),
                          interpolation=cv2.INTER_AREA)
        chars.append(crop)

    debug = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
    for i in range(1, n_chars):
        x = i * char_w
        cv2.line(debug, (x, 0), (x, h), (0, 255, 0), 2)

    return chars, debug

def predict_text(roi_bgr, bundle, label=''):
    gray = preprocess(roi_bgr)
    if gray is None:
        empty = np.zeros((10,10), np.uint8)
        return '', empty, cv2.cvtColor(empty, cv2.COLOR_GRAY2BGR)

    binary       = binarize(gray)
    n            = N_CHARS.get(label, 6)
    chars, debug = segment_fixed(binary, n)

    feats  = np.array([extract_hog(c) for c in chars])
    labels = bundle['clf'].predict(feats)
    known  = set(bundle['chars'])
    result = ''.join(l for l in labels if l in known)
    return result, binary, debug

def postprocess(label, text):
    text = text.strip()
    if label == 'NAMA_MATA_KULIAH':
        text = re.sub(r'^[^A-Za-z]+', '', text)
        corrections = {
            'Pomputer': 'Computer', 'Gomputer': 'Computer',
            'Yision'  : 'Vision',
        }
        for wrong, right in corrections.items():
            text = re.sub(wrong, right, text, flags=re.IGNORECASE)
        text = text.title()
    elif label in ('KODE_KELAS', 'RUANGAN'):
        text = text.replace(' ', '').upper()
    elif label == 'NO_MEJA':
        text = ''.join(re.findall(r'\d+', text))
    return text


bundle = load_or_train()
print(f'\nModel siap -> {len(bundle["chars"])} karakter')

ocr_results = {}
n   = len(HANDWRITING_ROIS)
fig, axes = plt.subplots(n, 4, figsize=(20, 3.2*n))
if n == 1: axes = [axes]

for i, (label, x1, y1, x2, y2) in enumerate(HANDWRITING_ROIS):
    roi_bgr          = warped[y1:y2, x1:x2]
    raw, binary, dbg = predict_text(roi_bgr, bundle, label)
    text             = postprocess(label, raw)
    ocr_results[label] = text

    b, g, r = cv2.split(roi_bgr.astype(np.float32))
    r_prev  = np.clip(r*0.8+g*0.15+b*0.05, 0, 255).astype(np.uint8)

    axes[i][0].imshow(cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f'ROI: {label}', fontsize=9, fontweight='bold')
    axes[i][0].axis('off')

    axes[i][1].imshow(binary, cmap='gray')
    axes[i][1].set_title('Binary', fontsize=9)
    axes[i][1].axis('off')

    axes[i][2].imshow(cv2.cvtColor(dbg, cv2.COLOR_BGR2RGB))
    axes[i][2].set_title(
        f'Fixed seg ({N_CHARS.get(label,6)} char)', fontsize=9)
    axes[i][2].axis('off')

    axes[i][3].imshow(r_prev, cmap='gray')
    axes[i][3].set_title(f'raw:"{raw}" -> "{text}"', fontsize=9)
    axes[i][3].axis('off')

plt.tight_layout()
plt.show()

print('\n' + '='*55)
print('  HASIL OCR - SVM + EMNIST ByClass')
print('='*55)
for label, text in ocr_results.items():
    print(f'  {label:<20}: {text}')

## 11. Scoring + Export Excel

In [ ]:
# ================================================
#  INPUT KUNCI JAWABAN
# ================================================
print('Masukkan kunci jawaban (contoh: ABCDE...)')
kunci_input = input('  Kunci jawaban: ').strip().upper()
kunci = {i+1: kunci_input[i] for i in range(len(kunci_input))}

# ================================================
#  HITUNG BENAR / SALAH / SCORE
# ================================================
benar = 0
salah = 0

for q, jawaban_siswa in all_answers.items():
    if q in kunci:
        if jawaban_siswa == kunci[q]:
            benar += 1
        else:
            salah += 1

total  = benar + salah
score  = round((benar / total) * 100, 2) if total > 0 else 0

print('\n' + '='*55)
print('  HASIL SCORING')
print('='*55)
print(f'  Nama (scan) : {nama_str}')
print(f'  NIM         : {nim_final}')
print(f'  Benar       : {benar}')
print(f'  Salah       : {salah}')
print(f'  Score       : {score}')

# ================================================
#  EXPORT EXCEL
# ================================================
import openpyxl
from google.colab import files

wb = openpyxl.Workbook()

# Sheet 1 - Hasil Scan
ws1 = wb.active
ws1.title = 'Hasil Scan'
ws1.append(['Nama', 'Benar', 'Salah', 'Score'])
ws1.append([nama_str, benar, salah, score])

# Sheet 2 - EDA placeholder
ws2 = wb.create_sheet('EDA')
ws2.append(['Fitur', 'Status'])
ws2.append(['EDA', 'Belum tersedia'])

wb.save('hasil_scan.xlsx')
print('\nFile Excel tersimpan: hasil_scan.xlsx')
files.download('hasil_scan.xlsx')

## 12. EDA
> **Fitur ini belum tersedia.** Akan dikerjakan pada iterasi berikutnya.